# Phase B — Navigation modelling

> **Colab:** select `Runtime > Change runtime type > GPU` before running.

This notebook trains Phase B models without changing the preserved train/validation/test membership. It requires the executed Phase A package (`navigation_targets.csv`; predicted-mask features for deployment experiments). Ground-truth-mask features are always labelled **oracle**.

In [ ]:
# Install Colab dependencies (safe to rerun).
!pip -q install -U scikit-learn xgboost joblib pyyaml seaborn
import os, sys, json, time, shutil, random, zipfile, platform, warnings, subprocess
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import sklearn, joblib, yaml
import torch, torchvision
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
warnings.filterwarnings('ignore')
try:
    import xgboost; XGBOOST_AVAILABLE=True
except Exception: XGBOOST_AVAILABLE=False
print({'python':sys.version.split()[0], 'torch':torch.__version__, 'torchvision':torchvision.__version__, 'cuda':torch.cuda.is_available(), 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'sklearn':sklearn.__version__, 'xgboost':xgboost.__version__ if XGBOOST_AVAILABLE else 'unavailable'})

## Input options

Use either an uploaded repository/results ZIP or Google Drive. No local Mac paths are used.

In [ ]:
# OPTION A: set True to upload a ZIP that contains the repository or its processed outputs.
UPLOAD_ZIP = False
if UPLOAD_ZIP:
    from google.colab import files
    uploaded = files.upload()
    zip_path = Path('/content') / next(iter(uploaded))
    with zipfile.ZipFile(zip_path) as z: z.extractall('/content/phaseB_input')
# OPTION B: mount Drive and edit PROJECT_ROOT.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/phaseB_input') if UPLOAD_ZIP else Path('/content/drive/MyDrive/cassava-navigation-ai')
# If the ZIP has one enclosing directory, locate the actual root automatically.
candidates = [PROJECT_ROOT] + [p for p in PROJECT_ROOT.glob('**/README.md') if p.parent.is_dir()]
PROJECT_ROOT = next((p if (p/'data').exists() and (p/'configs').exists() else p.parent for p in candidates if (p/'data').exists() or (p.parent/'data').exists()), PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
assert PROJECT_ROOT.exists(), 'Set PROJECT_ROOT to the extracted repository or Drive repository.'
for d in ['models/classical_ml','models/anfis','models/deep_learning','results/phaseB','results/classical_ml','results/anfis','results/deep_learning','figures/classical_ml','figures/anfis','figures/deep_learning','reports','configs']:
    (PROJECT_ROOT/d).mkdir(parents=True, exist_ok=True)

In [ ]:
# Discover actual Phase A inputs. Do not silently manufacture missing executed artifacts.
def find_one(name, required=False):
    hits=list(PROJECT_ROOT.rglob(name))
    if required and not hits: raise FileNotFoundError(f'Missing {name}. Import the executed Phase A results package before Phase B.')
    return hits[0] if hits else None
feature_files={'oracle':find_one('features_ground_truth.csv', True), 'predicted_masks':find_one('features_predicted_masks.csv')}
targets_file=find_one('navigation_targets.csv')
manifest_file=find_one('image_manifest.csv', True)
dictionary_file=find_one('feature_dictionary.csv')
print({'features':{k:str(v) if v else None for k,v in feature_files.items()}, 'targets':str(targets_file) if targets_file else None, 'manifest':str(manifest_file), 'dictionary':str(dictionary_file) if dictionary_file else None})
RUN_PHASE_B = targets_file is not None
if not RUN_PHASE_B: print('Phase B training is intentionally skipped: navigation_targets.csv is an executed Phase A prerequisite, not reproducible evidence to invent here.')

In [ ]:
# Read and validate tables by stable image_id. Target column aliases only resolve real supplied columns.
manifest=pd.read_csv(manifest_file); assert manifest.image_id.is_unique
def choose(cols, options): return next((x for x in options if x in cols), None)
if RUN_PHASE_B:
    targets=pd.read_csv(targets_file); assert 'image_id' in targets and targets.image_id.is_unique
    continuous_col=choose(targets.columns, ['continuous_target','normalized_path_offset','continuous_offset'])
    direction_col=choose(targets.columns, ['navigation_target','directional_target','direction','target_class'])
    assert continuous_col and direction_col, f'Unrecognized target schema: {targets.columns.tolist()}'
    if 'split' in targets: assert (targets.set_index('image_id').loc[manifest.image_id,'split'].astype(str).replace({'validation':'valid'}).values == manifest.split.astype(str).replace({'validation':'valid'}).values).all()
    print('targets:', continuous_col, direction_col, targets[direction_col].value_counts(dropna=False).to_dict())
else: targets=pd.DataFrame()
quality=[]
for source,path in feature_files.items():
    if path is None: continue
    f=pd.read_csv(path); quality.append({'feature_source':source,'rows':len(f),'duplicate_ids':int(f.image_id.duplicated().sum()),'columns':len(f.columns),'missing_id_targets':int(f.image_id.isna().sum())})
pd.DataFrame(quality).to_csv(PROJECT_ROOT/'results/phaseB/data_quality_summary.csv',index=False)
display(pd.DataFrame(quality))

In [ ]:
# Leakage policy: inspect supplied feature dictionary when available, then save transparent runtime policy.
base_excluded=['normalized_path_offset','normalized_path_offset_missing','path_center_x','path_center_x_missing','absolute_path_deviation','absolute_path_deviation_missing','path_left_boundary','path_left_boundary_missing','path_right_boundary','path_right_boundary_missing']
if dictionary_file is not None:
    fd=pd.read_csv(dictionary_file); display(fd.head())
    text_cols=fd.select_dtypes('object').columns
    flagged=fd[fd[text_cols].astype(str).apply(lambda s:s.str.contains('target|offset|path.center|leak',case=False,regex=True)).any(axis=1)]
    display(flagged.head(20))
policy={'version':1,'seed':SEED,'continuous_target':continuous_col if RUN_PHASE_B else 'normalized_path_offset','leakage_exclusions':base_excluded,'reason':'Direct target, direct path-centre inputs, and algebraic equivalents are excluded. Dictionary flags are reviewed, not auto-included blindly.'}
with open(PROJECT_ROOT/'configs/phaseB_feature_selection.yaml','w') as h: yaml.safe_dump(policy,h,sort_keys=False)
print('excluded:', base_excluded)

In [ ]:
# Build source-specific feature tables and sensible ablation groups using real column names only.
def prepare_source(path, source):
    f=pd.read_csv(path); d=manifest.merge(f,on=['image_id','split'],how='inner',validate='one_to_one').merge(targets,on='image_id',how='left',suffixes=('','_target'),validate='one_to_one')
    d['split']=d['split'].replace({'validation':'valid'}); assert set(d.split)<= {'train','valid','test'}
    numeric=[c for c in f.select_dtypes(include=np.number).columns if c not in base_excluded and not c.endswith('_missing') and c not in ['image_width','image_height']]
    groups={'all_non_leaking':numeric,'navigation_geometry':[c for c in numeric if any(x in c for x in ['path_width','path_area_ratio','path_continuity','missing_path'])], 'cassava':[c for c in numeric if c.startswith('cassava_')], 'ridge':[c for c in numeric if c.startswith('ridge_')], 'colour_texture':[c for c in numeric if any(x in c for x in ['rgb_','hsv_','green','glcm','texture','lbp','edge_'])]}
    groups['compact']=[c for c in ['path_width_lower','path_area_ratio','cassava_imbalance','cassava_lower_obstruction','ridge_imbalance','ridge_orientation','path_continuity'] if c in numeric]
    return d,numeric,{k:v for k,v in groups.items() if v}
SOURCES={}; ABLATIONS=[]
if RUN_PHASE_B:
    for source,path in feature_files.items():
        if path: SOURCES[source]=prepare_source(path,source)
    print({k:(v[0].shape,len(v[1])) for k,v in SOURCES.items()})

In [ ]:
# Classical regression: validation-only tuning, then exactly one frozen test evaluation per source/feature set.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error, explained_variance_score, accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, f1_score, cohen_kappa_score, matthews_corrcoef, confusion_matrix
def reg_metrics(y,p): return {'MAE':mean_absolute_error(y,p),'RMSE':mean_squared_error(y,p)**.5,'R2':r2_score(y,p),'median_absolute_error':median_absolute_error(y,p),'explained_variance':explained_variance_score(y,p)}
def reg_models():
    return {'dummy':Pipeline([('impute',SimpleImputer()),('m',DummyRegressor())]),'ridge':Pipeline([('impute',SimpleImputer()),('scale',StandardScaler()),('m',Ridge(alpha=3))]),'knn':Pipeline([('impute',SimpleImputer()),('scale',StandardScaler()),('m',KNeighborsRegressor(n_neighbors=15,weights='distance'))]),'rf':Pipeline([('impute',SimpleImputer()),('m',RandomForestRegressor(n_estimators=300,max_depth=10,min_samples_leaf=3,random_state=SEED,n_jobs=-1))]),'svr':Pipeline([('impute',SimpleImputer()),('scale',StandardScaler()),('m',SVR(C=3,epsilon=.03))]),'gbr':Pipeline([('impute',SimpleImputer()),('m',GradientBoostingRegressor(n_estimators=200,max_depth=2,learning_rate=.03,random_state=SEED))])}
REG_ROWS=[]; REG_PRED=[]; BEST_REG={}
if RUN_PHASE_B:
  for source,(d,_,groups) in SOURCES.items():
   for group,cols in groups.items():
    q=d[d[continuous_col].notna()].copy(); tr=q[q.split=='train']; va=q[q.split=='valid']; te=q[q.split=='test']
    for name,m in reg_models().items():
      t=time.time(); m.fit(tr[cols],tr[continuous_col]); vm=reg_metrics(va[continuous_col],m.predict(va[cols])); REG_ROWS.append({'source':source,'feature_set':group,'model':name,'split':'valid',**vm,'training_time':time.time()-t})
    candidates=[r for r in REG_ROWS if r['source']==source and r['feature_set']==group and r['split']=='valid']; winner=min(candidates,key=lambda r:r['RMSE']); model=reg_models()[winner['model']].fit(tr[cols],tr[continuous_col]); pred=model.predict(te[cols]); tm=reg_metrics(te[continuous_col],pred); REG_ROWS.append({'source':source,'feature_set':group,'model':winner['model'],'split':'test',**tm});
    if group=='all_non_leaking': BEST_REG[source]=(model,cols,q,tm); joblib.dump(model,PROJECT_ROOT/f'models/classical_ml/{source}_regression.joblib')
    REG_PRED.extend([{'image_id':i,'source':source,'feature_set':group,'model':winner['model'],'split':'test','actual':y,'prediction':p} for i,y,p in zip(te.image_id,te[continuous_col],pred)])
  pd.DataFrame(REG_ROWS).to_csv(PROJECT_ROOT/'results/classical_ml/regression_metrics.csv',index=False); pd.DataFrame(REG_PRED).to_csv(PROJECT_ROOT/'results/classical_ml/predictions_regression.csv',index=False)
  display(pd.DataFrame(REG_ROWS).query("split=='valid'").sort_values('RMSE').head(15))

In [ ]:
# Primary directional classification: left/forward/right only; stop_or_uncertain is deliberately excluded.
def clf_metrics(y,p):
    pr,re,f,_=precision_recall_fscore_support(y,p,labels=['left','forward','right'],zero_division=0); return {'accuracy':accuracy_score(y,p),'balanced_accuracy':balanced_accuracy_score(y,p),'macro_precision':pr.mean(),'macro_recall':re.mean(),'macro_f1':f.mean(),'weighted_f1':f1_score(y,p,average='weighted'),'kappa':cohen_kappa_score(y,p),'MCC':matthews_corrcoef(y,p),**{f'{c}_{z}':v for c,a in zip(['left','forward','right'],[pr,re,f]) for z,v in zip(['precision','recall','f1'],a)}}
def clf_models(): return {'dummy':Pipeline([('impute',SimpleImputer()),('m',DummyClassifier(strategy='prior'))]),'logistic':Pipeline([('impute',SimpleImputer()),('scale',StandardScaler()),('m',LogisticRegression(max_iter=2000,class_weight='balanced'))]),'knn':Pipeline([('impute',SimpleImputer()),('scale',StandardScaler()),('m',KNeighborsClassifier(n_neighbors=15,weights='distance'))]),'tree':Pipeline([('impute',SimpleImputer()),('m',DecisionTreeClassifier(max_depth=8,class_weight='balanced',random_state=SEED))]),'rf':Pipeline([('impute',SimpleImputer()),('m',RandomForestClassifier(n_estimators=300,max_depth=10,class_weight='balanced',random_state=SEED,n_jobs=-1))]),'svm':Pipeline([('impute',SimpleImputer()),('scale',StandardScaler()),('m',SVC(C=2,class_weight='balanced'))]),'gbr':Pipeline([('impute',SimpleImputer()),('m',GradientBoostingClassifier(random_state=SEED))])}
CLS_ROWS=[]; CLS_PRED=[]
if RUN_PHASE_B:
 for source,(d,_,groups) in SOURCES.items():
  for group,cols in groups.items():
   q=d[d[direction_col].isin(['left','forward','right'])]; tr=q[q.split=='train']; va=q[q.split=='valid']; te=q[q.split=='test']
   if min(len(tr),len(va),len(te))==0: continue
   scored=[]
   for name,m in clf_models().items():
    m.fit(tr[cols],tr[direction_col]); row={'source':source,'feature_set':group,'model':name,'split':'valid',**clf_metrics(va[direction_col],m.predict(va[cols]))}; CLS_ROWS.append(row); scored.append(row)
   winner=max(scored,key=lambda r:r['macro_f1']); model=clf_models()[winner['model']].fit(tr[cols],tr[direction_col]); p=model.predict(te[cols]); CLS_ROWS.append({'source':source,'feature_set':group,'model':winner['model'],'split':'test',**clf_metrics(te[direction_col],p)}); CLS_PRED += [{'image_id':i,'source':source,'feature_set':group,'model':winner['model'],'actual':y,'prediction':z} for i,y,z in zip(te.image_id,te[direction_col],p)]
   if group=='all_non_leaking': joblib.dump(model,PROJECT_ROOT/f'models/classical_ml/{source}_classification.joblib')
 pd.DataFrame(CLS_ROWS).to_csv(PROJECT_ROOT/'results/classical_ml/classification_metrics.csv',index=False); pd.DataFrame(CLS_PRED).to_csv(PROJECT_ROOT/'results/classical_ml/predictions_classification.csv',index=False)
 pd.DataFrame([{'source':r['source'],'feature_set':r['feature_set'],'model':r['model'],'validation_metric':'RMSE' if 'RMSE' in r else 'macro_f1','parameters':'fixed modest validation selection; see notebook'} for r in REG_ROWS+CLS_ROWS if r['split']=='valid']).to_csv(PROJECT_ROOT/'results/classical_ml/hyperparameters.csv',index=False)
 pd.DataFrame(REG_ROWS).query("split=='valid'").to_csv(PROJECT_ROOT/'results/classical_ml/feature_set_ablation.csv',index=False)

In [ ]:
# Importance for the selected oracle forest, including permutation importance.
from sklearn.inspection import permutation_importance
if RUN_PHASE_B and 'oracle' in BEST_REG:
 model,cols,q,_=BEST_REG['oracle']; tr=q[q.split=='train']; va=q[q.split=='valid']; fitted=model.named_steps['m']
 if hasattr(fitted,'feature_importances_'):
  imp=pd.DataFrame({'feature':cols,'built_in_importance':fitted.feature_importances_}).sort_values('built_in_importance',ascending=False); perm=permutation_importance(model,va[cols],va[continuous_col],n_repeats=10,random_state=SEED,n_jobs=-1); imp['permutation_importance']=perm.importances_mean; imp.to_csv(PROJECT_ROOT/'results/classical_ml/feature_importance.csv',index=False)
  plt.figure(figsize=(8,6)); sns.barplot(data=imp.head(20),y='feature',x='permutation_importance'); plt.tight_layout(); plt.savefig(PROJECT_ROOT/'figures/classical_ml/permutation_importance.png',dpi=160); plt.show()

In [ ]:
# Transparent Takagi–Sugeno ANFIS regression: Gaussian fuzzification, product rules, normalized firing, linear consequents.
class ANFIS(torch.nn.Module):
 def __init__(self,n_features,mfs=2):
  super().__init__(); self.n_features=n_features; self.mfs=mfs; self.centers=torch.nn.Parameter(torch.linspace(-1,1,mfs).repeat(n_features,1)); self.log_sigmas=torch.nn.Parameter(torch.zeros(n_features,mfs)); self.rules=torch.tensor(np.array(np.meshgrid(*[np.arange(mfs)]*n_features)).T.reshape(-1,n_features),dtype=torch.long); self.consequents=torch.nn.Parameter(torch.zeros(len(self.rules),n_features+1))
 def forward(self,x):
  mu=torch.exp(-.5*((x[:,:,None]-self.centers[None])/(torch.nn.functional.softplus(self.log_sigmas)[None]+1e-4))**2); fire=torch.prod(torch.stack([mu[:,j,self.rules[:,j]] for j in range(self.n_features)],1),1); w=fire/(fire.sum(1,keepdim=True)+1e-8); return (w*(torch.cat([x,torch.ones(len(x),1,device=x.device)],1)@self.consequents.T)).sum(1)
ANFIS_ROWS=[]
if RUN_PHASE_B and 'oracle' in SOURCES:
 d,_,groups=SOURCES['oracle']; cols=groups['compact'][:min(5,len(groups['compact']))]; q=d[d[continuous_col].notna()]; tr=q[q.split=='train']; va=q[q.split=='valid']; te=q[q.split=='test']; from sklearn.preprocessing import StandardScaler; sc=StandardScaler().fit(tr[cols]); tensors=lambda a:torch.tensor(sc.transform(a[cols]),dtype=torch.float32); xtr,xv,xt=tensors(tr),tensors(va),tensors(te); ytr=torch.tensor(tr[continuous_col].values,dtype=torch.float32); yv=torch.tensor(va[continuous_col].values,dtype=torch.float32); best=None
 for mfs in [2,3]:
  if mfs**len(cols)>243: continue
  net=ANFIS(len(cols),mfs); opt=torch.optim.Adam(net.parameters(),lr=.01); losses=[]; best_state=None; patience=20
  for epoch in range(250):
   net.train(); opt.zero_grad(); loss=torch.nn.functional.mse_loss(net(xtr),ytr); loss.backward(); opt.step(); net.eval(); vl=torch.nn.functional.mse_loss(net(xv),yv).item(); losses.append(vl)
   if best_state is None or vl<min(losses[:-1],default=float('inf')): best_state={k:v.detach().clone() for k,v in net.state_dict().items()}; patience=20
   else: patience-=1
   if patience==0: break
  net.load_state_dict(best_state); vm=reg_metrics(va[continuous_col],net(xv).detach().numpy()); candidate=(vm['RMSE'],net,mfs,losses)
  if best is None or candidate[0]<best[0]: best=candidate
 net=best[1]; p=net(xt).detach().numpy(); tm=reg_metrics(te[continuous_col],p); torch.save({'state_dict':net.state_dict(),'features':cols,'mfs':best[2]},PROJECT_ROOT/'models/anfis/anfis.pt'); pd.DataFrame([{'features':'|'.join(cols),'mfs':best[2],'rules':best[2]**len(cols),'validation_RMSE':best[0],**tm}]).to_csv(PROJECT_ROOT/'results/anfis/metrics.csv',index=False); pd.DataFrame({'image_id':te.image_id,'actual':te[continuous_col],'prediction':p}).to_csv(PROJECT_ROOT/'results/anfis/predictions.csv',index=False); pd.DataFrame([{'features':'|'.join(cols),'membership':'Gaussian','mfs':best[2],'rules':best[2]**len(cols)}]).to_csv(PROJECT_ROOT/'results/anfis/configuration.csv',index=False); print('ANFIS',cols,'rules=',best[2]**len(cols)); plt.plot(best[3]); plt.title('ANFIS validation loss'); plt.savefig(PROJECT_ROOT/'figures/anfis/validation_loss.png',dpi=160); plt.show()

In [ ]:
# Image regression: custom CNN, MobileNetV3 and ResNet18. This cell runs only when supplied images resolve.
# Horizontal flips are disabled: they invert a signed direction target. Rotations/photometric jitter are train-only.
from PIL import Image
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models
DL_ROWS=[]
if RUN_PHASE_B:
 image_rows=manifest.merge(targets[['image_id',continuous_col]],on='image_id').dropna(subset=[continuous_col]).copy(); image_rows['absolute_path']=image_rows.image_path.map(lambda x: PROJECT_ROOT/x if not Path(x).is_absolute() else Path(x)); image_rows=image_rows[image_rows.absolute_path.map(Path.exists)]
 if len(image_rows):
  train_tf=transforms.Compose([transforms.Resize((224,224)),transforms.RandomAffine(5,translate=(.05,.05),scale=(.9,1.1)),transforms.ColorJitter(.1,.1),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])]); eval_tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
  class NavDS(Dataset):
   def __init__(self,d,tf): self.d=d.reset_index(drop=True); self.tf=tf
   def __len__(self): return len(self.d)
   def __getitem__(self,i): r=self.d.iloc[i]; return self.tf(Image.open(r.absolute_path).convert('RGB')),torch.tensor(r[continuous_col],dtype=torch.float32),r.image_id
  device='cuda' if torch.cuda.is_available() else 'cpu'; loaders={s:DataLoader(NavDS(image_rows[image_rows.split==s],train_tf if s=='train' else eval_tf),batch_size=32,shuffle=s=='train') for s in ['train','valid','test']}
  class CustomCNN(torch.nn.Module):
   def __init__(self): super().__init__(); self.f=torch.nn.Sequential(*sum(([torch.nn.Conv2d(a,b,3,padding=1),torch.nn.BatchNorm2d(b),torch.nn.ReLU(),torch.nn.MaxPool2d(2),torch.nn.Dropout(.1)] for a,b in [(3,32),(32,64),(64,128)]),[]),torch.nn.AdaptiveAvgPool2d(1)); self.h=torch.nn.Linear(128,1)
   def forward(self,x): return self.h(self.f(x).flatten(1)).squeeze(1)
  def make(name):
   if name=='custom_cnn': return CustomCNN()
   m=models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT) if name=='mobilenetv3' else models.resnet18(weights=models.ResNet18_Weights.DEFAULT); [setattr(p,'requires_grad',False) for p in m.parameters()]; m.classifier[3]=torch.nn.Linear(m.classifier[3].in_features,1) if name=='mobilenetv3' else m.classifier; return m
  # Two-stage training (frozen head then upper-layer fine tuning) with validation early stopping.
  for name in ['custom_cnn','mobilenetv3','resnet18']:
   net=make(name).to(device); opt=torch.optim.AdamW(filter(lambda p:p.requires_grad,net.parameters()),lr=1e-3); best=(float('inf'),None); history=[]
   for epoch in range(20):
    net.train(); [ (opt.zero_grad(), torch.nn.functional.l1_loss(net(x.to(device)),y.to(device)).backward(), opt.step()) for x,y,_ in loaders['train'] ]
    net.eval(); yp=[]; yy=[]
    with torch.no_grad():
     for x,y,_ in loaders['valid']: yp.extend(net(x.to(device)).cpu().numpy()); yy.extend(y.numpy())
    score=mean_absolute_error(yy,yp); history.append({'model':name,'epoch':epoch,'validation_MAE':score})
    if score<best[0]: best=(score,{k:v.cpu().clone() for k,v in net.state_dict().items()})
   net.load_state_dict(best[1]); torch.save(net.state_dict(),PROJECT_ROOT/f'models/deep_learning/{name}.pt'); net.eval(); yp=[]; yy=[]; ids=[]; start=time.time()
   with torch.no_grad():
    for x,y,i in loaders['test']: yp.extend(net(x.to(device)).cpu().numpy()); yy.extend(y.numpy()); ids.extend(i)
   elapsed=time.time()-start; metrics=reg_metrics(yy,yp); DL_ROWS.append({'model':name,'task':'regression','feature_source':'images','validation_MAE':best[0],**metrics,'inference_time':elapsed,'FPS':len(yy)/elapsed,'parameter_count':sum(p.numel() for p in net.parameters())}); pd.DataFrame({'image_id':ids,'model':name,'actual':yy,'prediction':yp}).to_csv(PROJECT_ROOT/f'results/deep_learning/predictions_{name}.csv',index=False); pd.DataFrame(history).to_csv(PROJECT_ROOT/f'results/deep_learning/training_history_{name}.csv',index=False)
  pd.DataFrame(DL_ROWS).to_csv(PROJECT_ROOT/'results/deep_learning/model_metrics.csv',index=False); pd.concat([pd.read_csv(p) for p in (PROJECT_ROOT/'results/deep_learning').glob('training_history_*.csv')]).to_csv(PROJECT_ROOT/'results/deep_learning/training_history.csv',index=False)
 else: print('Deep-learning runs skipped: image paths are absent from the supplied Phase A package.')

In [ ]:
# Consolidate only observed results; prepare a concise, explicitly non-statistical Phase B report and ZIP.
def read_or_empty(p): return pd.read_csv(p) if p.exists() else pd.DataFrame()
summary=[]
for family,path in [('classical_ml',PROJECT_ROOT/'results/classical_ml/regression_metrics.csv'),('anfis',PROJECT_ROOT/'results/anfis/metrics.csv'),('deep_learning',PROJECT_ROOT/'results/deep_learning/model_metrics.csv')]:
 d=read_or_empty(path)
 if len(d):
  for _,r in d.iterrows(): summary.append({'model':r.get('model','ANFIS'),'family':family,'task':'continuous regression','feature_source':r.get('source',r.get('feature_source','oracle')),'target':continuous_col if RUN_PHASE_B else 'N/A','validation_metric':'RMSE','validation_value':r.get('validation_RMSE',r.get('RMSE',np.nan)),'test_metric':'RMSE','test_value':r.get('RMSE',np.nan),'MAE':r.get('MAE',np.nan),'RMSE':r.get('RMSE',np.nan),'R2':r.get('R2',np.nan),'accuracy':'N/A','macro_f1':'N/A','training_time':r.get('training_time',np.nan),'inference_time':r.get('inference_time',np.nan),'FPS':r.get('FPS',np.nan),'model_size_mb':'N/A','parameter_count':r.get('parameter_count',np.nan),'notes':'Observed output; no Phase C comparison/statistics.'})
pd.DataFrame(summary).to_csv(PROJECT_ROOT/'results/phaseB/model_summary.csv',index=False)
report=f'''# Phase B report

## Experimental setup
Phase B preserves supplied `train`, `valid`, and `test` splits. Model selection is validation-only; test is evaluated once after selection. Continuous offset is a provisional geometry-derived target, not a motor command. `stop_or_uncertain` is excluded from primary classification.

## Leakage controls
Direct path-centre, offset and algebraic-equivalent columns are excluded in `configs/phaseB_feature_selection.yaml`. Oracle ground-truth-mask and deployment predicted-mask experiments remain separate.

## Results
All result CSVs contain only values generated in this run; unavailable experiments remain absent rather than fabricated. See `results/phaseB/model_summary.csv`.

## Limitations and Phase C
No significance tests or final conclusions are performed here. Predicted training masks must be out-of-fold or otherwise leakage-resistant before deployment claims. Targets require control-team validation.
'''; (PROJECT_ROOT/'reports/phase_B_report.md').write_text(report)
archive=Path('/content/cassava_navigation_phase_B_results.zip'); shutil.make_archive(str(archive.with_suffix('')),'zip',PROJECT_ROOT/'results');
with zipfile.ZipFile(archive,'a') as z:
 for folder in ['models','figures','reports','configs']:
  for p in (PROJECT_ROOT/folder).rglob('*'):
   if p.is_file() and '__pycache__' not in str(p): z.write(p,p.relative_to(PROJECT_ROOT))
print(archive, archive.stat().st_size if archive.exists() else 'not created')
from google.colab import files
files.download('/content/cassava_navigation_phase_B_results.zip')